In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))  # adds /home/patroklos/delphi.ai to the path

import torch as ch 
from torch.distributions import Gumbel
from torch.nn import CosineSimilarity
from torch.utils.data import DataLoader, TensorDataset
from torch.nn import Softmax, CrossEntropyLoss
from torch.optim import SGD
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

from delphi.utils.helpers import Parameters
from delphi.utils.datasets import make_train_and_val
from delphi.grad import TruncatedCE

In [2]:
cos_sim = CosineSimilarity()
gumbel = Gumbel(0, 1)
softmax = Softmax(dim=1)
ce = CrossEntropyLoss()
trunc_ce = TruncatedCE.apply 

def phi(z): 
    return c.ones_like()

In [52]:
ch.ones_like(ch.randn(10, 1))

tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.]])

In [22]:
D, K = 3, 2

W = ch.rand(K, D)
W_eff = W[1] - W[0]
print(f'ground truth: {W}')
print(f'effective ground truth: {W_eff}')

# input features
NUM_SAMPLES = 5000
X = ch.rand(NUM_SAMPLES, D) 
# latent variables
z = X@W.T + gumbel.sample([X.size(0), K])
# classification
y = z.argmax(-1, keepdim=True)

sklearn = LogisticRegression(penalty=None, fit_intercept=False)
sklearn.fit(X, y.flatten())
sklearn_ = ch.from_numpy(sklearn.coef_)

print(f'sklearn: {sklearn_}')
pred = sklearn.predict(X)
acc = np.equal(pred, y.flatten()).sum() / len(y)
print(f'sklearn acc: {acc}')
sklearn_conf_matrix = confusion_matrix(y, pred)
print(f'sklearn confusion matrix: \n {sklearn_conf_matrix}')

ground truth: tensor([[0.9321, 0.1566, 0.1939],
        [0.5611, 0.7104, 0.4340]])
effective ground truth: tensor([-0.3710,  0.5538,  0.2401])
sklearn: tensor([[-0.2421,  0.4879,  0.1419]], dtype=torch.float64)
sklearn acc: 0.5541999936103821
sklearn confusion matrix: 
 [[ 335 1927]
 [ 302 2436]]


/tmp/ipykernel_220710/43314962.py:22: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  acc = np.equal(pred, y.flatten()).sum() / len(y)


In [23]:
epochs = 50 
batch_size = 50

val = int(.2 * X.size(0))

X_train,y_train = X[val:], y[val:]
X_val, y_val = X[:val], y[:val]


train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=batch_size, num_workers=1, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=X_val.size(0), num_workers=1, shuffle=True)

In [24]:
ch.manual_seed(69)
weight = ch.nn.Parameter(ch.randn(K, D))
optim = SGD([weight], lr=1e-1)

best_weight, best_loss = None, None


for i in range(epochs): 
    for batch in train_loader: 
        optim.zero_grad()
        x_, y_ = batch 
        
        logits = x_@weight.T 
        pred = softmax(logits)
        
        loss = trunc_ce(logits, y_, phi)
        loss = loss.sum()
        
        if best_weight is None or loss < best_loss: 
            best_loss = loss 
            best_weight = weight.clone().detach()
        
        
        loss.backward()
        
        optim.step()
    print(f'Epoch {i+1}, loss={loss.item()}')
        
weight.requires_grad = False

Epoch 1, loss=-3.8825511932373047
Epoch 2, loss=-3.9118051528930664
Epoch 3, loss=-3.922694206237793
Epoch 4, loss=-3.931753635406494
Epoch 5, loss=-3.9219937324523926
Epoch 6, loss=-3.8686323165893555
Epoch 7, loss=-3.9084630012512207
Epoch 8, loss=-3.92393159866333
Epoch 9, loss=-3.9038825035095215
Epoch 10, loss=-3.8704888820648193
Epoch 11, loss=-3.9155657291412354
Epoch 12, loss=-3.909097909927368
Epoch 13, loss=-3.904393196105957
Epoch 14, loss=-3.9304652214050293
Epoch 15, loss=-3.8740265369415283
Epoch 16, loss=-3.945309638977051
Epoch 17, loss=-3.9251129627227783
Epoch 18, loss=-3.906092405319214
Epoch 19, loss=-3.9145681858062744
Epoch 20, loss=-3.8928520679473877
Epoch 21, loss=-3.90877103805542
Epoch 22, loss=-3.9146056175231934
Epoch 23, loss=-3.9057445526123047
Epoch 24, loss=-3.915905714035034
Epoch 25, loss=-3.9029057025909424
Epoch 26, loss=-3.9120736122131348
Epoch 27, loss=-3.9030075073242188
Epoch 28, loss=-3.913924217224121
Epoch 29, loss=-3.913135051727295
Epoch 3

In [25]:
print(f'estimated weight:\n {best_weight.data.tolist()}')
best_weight_diff_ = best_weight[1] - best_weight[0]
print(f'estimated weight diff:\n {best_weight_diff_.data.tolist()}')

estimated weight:
 [[0.7701139450073242, -2.2471818923950195, -0.5599315762519836], [0.607056736946106, -1.4667264223098755, -0.3967365324497223]]
estimated weight diff:
 [-0.16305720806121826, 0.780455470085144, 0.16319504380226135]


In [26]:
soft_cos_sim = float(cos_sim(best_weight_diff_[None,...], W_eff))
pred = softmax(X@best_weight.T).argmax(-1)
soft_acc = pred.eq(y.flatten()).sum() / len(y)
print(f'softmax accuracy: {soft_acc}')
print(f'softmax cos sim: {soft_cos_sim}')
soft_conf_matrix = confusion_matrix(y, pred)
print(f'softmax confusion matrix: \n {soft_conf_matrix}')

softmax accuracy: 0.5490000247955322
softmax cos sim: 0.9224463105201721
softmax confusion matrix: 
 [[  92 2170]
 [  85 2653]]


In [27]:
ch.manual_seed(69)
weight = ch.nn.Parameter(ch.randn(K, D))
optim = SGD([weight], lr=1e-1)


best_weight, best_loss = None, None

for i in range(epochs): 
    for batch in train_loader: 
        optim.zero_grad()
        x_, y_ = batch 
        
        logits = x_@weight.T 
        pred = softmax(logits)
        
        loss = ce(logits, y_.flatten())
        loss = loss.sum()
        
        if best_weight is None or loss < best_loss: 
            best_loss = loss 
            best_weight = weight.clone().detach()
        
        loss.backward()
        
        optim.step()
    print(f'Epoch {i+1}, loss={loss.item()}')
        
weight.requires_grad = False

Epoch 1, loss=0.7320680022239685
Epoch 2, loss=0.6931484341621399
Epoch 3, loss=0.6427886486053467
Epoch 4, loss=0.6604436635971069
Epoch 5, loss=0.7184597849845886
Epoch 6, loss=0.6842032074928284
Epoch 7, loss=0.6920996308326721
Epoch 8, loss=0.6611955761909485
Epoch 9, loss=0.670318603515625
Epoch 10, loss=0.7149885296821594
Epoch 11, loss=0.6924140453338623
Epoch 12, loss=0.6937838792800903
Epoch 13, loss=0.6881504058837891
Epoch 14, loss=0.6732768416404724
Epoch 15, loss=0.6818881034851074
Epoch 16, loss=0.7128055095672607
Epoch 17, loss=0.7033844590187073
Epoch 18, loss=0.6637259125709534
Epoch 19, loss=0.6818760633468628
Epoch 20, loss=0.6949555277824402
Epoch 21, loss=0.6747866272926331
Epoch 22, loss=0.6866062879562378
Epoch 23, loss=0.648154079914093
Epoch 24, loss=0.6599840521812439
Epoch 25, loss=0.6891061663627625
Epoch 26, loss=0.6918571591377258
Epoch 27, loss=0.6438686847686768
Epoch 28, loss=0.6849335432052612
Epoch 29, loss=0.7009779214859009
Epoch 30, loss=0.70289075

In [28]:
print(f'estimated weight:\n {weight.data.tolist()}')
weight_diff_ = weight[1] - weight[0]
print(f'estimated weight diff:\n {weight_diff_.data.tolist()}')

estimated weight:
 [[0.5847402811050415, -1.8705185651779175, -0.7320941090583801], [0.32501712441444397, -1.3133357763290405, -0.638079822063446]]
estimated weight diff:
 [-0.25972315669059753, 0.557182788848877, 0.09401428699493408]


In [29]:
soft_cos_sim = float(cos_sim(weight_diff_[None,...], W_eff))
pred = softmax(X@weight.T).argmax(-1)
soft_acc = pred.eq(y.flatten()).sum() / len(y)
print(f'softmax accuracy: {soft_acc}')
print(f'softmax cos sim: {soft_cos_sim}')
soft_conf_matrix = confusion_matrix(y, pred)
print(f'softmax confusion matrix: \n {soft_conf_matrix}')

softmax accuracy: 0.5533999800682068
softmax cos sim: 0.9702335596084595
softmax confusion matrix: 
 [[ 403 1859]
 [ 374 2364]]


In [49]:
import torch as ch
from torch.distributions import Gumbel

gumbel = Gumbel(0., 1.)

class TruncatedCE(ch.autograd.Function):
    @staticmethod
    def forward(ctx, logits, targ, phi, num_samples=2000, eps=1e-8):
        """
        logits: (batch, K)  raw logits mu
        targ:   (batch, 1)  long
        phi:    callable phi(noised_logits, targ) -> bool tensor (S, batch, 1) or (S, batch)
        """
        # shapes
        batch, K = logits.shape
        device = logits.device
        mu = logits

        # fast path: if phi always returns ones, forward to CE and note fast path
        # (we detect by asking phi on a small number of samples)
        check_noised = mu[None,...] + gumbel.sample((min(64, num_samples),) + mu.shape).to(device)
        try:
            chk = phi(check_noised, targ)
        except TypeError:
            chk = phi(check_noised)
        if chk.all() and False:
            # store flag and just return standard CE (we still save tensors for backward)
            probs = ch.log_softmax(mu, dim=-1)
            idx = targ.squeeze(-1)
            ce_vals = -probs[range(batch), idx]
            ctx.save_for_backward(mu, targ)
            ctx.is_fast_path = True
            return ce_vals  # (batch,)
        ctx.is_fast_path = False

        # Monte-Carlo samples of z = mu + eps
        S = num_samples
        stacked = mu[None,...].expand(S, -1, -1)             # (S, batch, K)
        eps_samples = gumbel.sample(stacked.size()).to(device)   # (S, batch, K)
        noised = stacked + eps_samples                         # (S, batch, K)

        # labels under noisy draw
        noised_labs = noised.argmax(dim=-1, keepdim=True)     # (S, batch, 1)

        # support indicator phi (may depend on targ)
        try:
            filtered = phi(noised, targ).float()              # expected shape (S, batch, 1) or (S, batch)
        except TypeError:
            filtered = phi(noised).float()
        print(f'filtered size: {filtered.size()}')

        # ensure shape (S, batch, 1)
        if filtered.dim() == 2:
            filtered = filtered.unsqueeze(-1)

        mask = (noised_labs == targ) * 1.0                    # float (S, batch, 1)

        # numerator and denominator for prob estimate
        import pdb; pdb.set_trace()
        num = (mask * filtered).sum(dim=0)                    # (batch, 1)
        den = filtered.sum(dim=0)                             # (batch, 1)
        prob_est = (num + eps) / (den + eps)                  # (batch, 1)

        # Save tensors for backward estimator
        ctx.save_for_backward(eps_samples, mask, filtered, mu, targ)
        ctx.num_samples = S
        ctx.eps = eps

        # return per-sample negative log-prob (no reduction)
        return -ch.log(prob_est.squeeze(-1))                  # (batch,)

    @staticmethod
    def backward(ctx, grad_output):
        # grad_output: (batch,) upstream grad (possibly ones if .sum() was called)
        if getattr(ctx, 'is_fast_path', False) and False:
            mu, targ = ctx.saved_tensors
            probs = ch.softmax(mu, dim=-1)
            idx = targ.squeeze(-1)
            onehot = ch.nn.functional.one_hot(idx, num_classes=mu.size(-1)).float()
            grad_mu = (probs - onehot)                               # (batch, K)
            # match shape: multiply by grad_output[:,None]
            return grad_output.unsqueeze(1) * grad_mu, None, None, None, None

        eps_samples, mask, filtered, mu, targ = ctx.saved_tensors
        S = ctx.num_samples
        eps = ctx.eps
        batch = mu.size(0)
        device = mu.device

        # compute location-score s_mu for Gumbel: s = 1 - exp(-u) where u = eps (since z - mu = eps)
        # eps_samples shape: (S, batch, K) ; phi has shape (S,batch,1), mask (S,batch,1)
        s_mu = 1.0 - ch.exp(-eps_samples)   # (S, batch, K)

        # we need inner products with respect to mu per class.
        # For class-conditional expectation we only average over the class index where mask==1.
        # But mask indicates the whole sample is of the correct class (S,batch,1) and s_mu is per-class.
        # We convert s_mu to per-sample scalar by taking s_mu at the winning class for that sample before argmax?
        # However s_mu is vector valued (K-d) and gradient wrt mu is vector. We compute vector expectations:
        # For each class dimension j, use s_mu[..., j].
        # Multiply mask (S,batch,1) by s_mu (S,batch,K) broadcasting last dim.

        # numerator expectation (conditioned on y and I)
        # sum_a = sum_s a_s = (mask * filtered).sum(0)  -> (batch,1)
        sum_a = (mask * filtered).sum(dim=0) + eps    # (batch,1)
        # sum_b = sum_s b_s = filtered.sum(0) -> (batch,1)
        sum_b = (filtered).sum(dim=0) + eps          # (batch,1)

        # compute weighted sums of s_mu across S
        # shape (S,batch, K) * (S,batch,1) -> (S,batch,K)
        weighted_a = (mask * filtered) * s_mu        # (S,batch,K)
        weighted_b = (filtered) * s_mu               # (S,batch,K)

        # sums over S
        sum_weighted_a = weighted_a.sum(dim=0)       # (batch,K)
        sum_weighted_b = weighted_b.sum(dim=0)       # (batch,K)

        # conditional expectations
        E_cond_xy = sum_weighted_a / sum_a           # (batch,K) broadcasting sum_a
        E_cond_x  = sum_weighted_b / sum_b           # (batch,K)

        # truncation correction gradient wrt mu: (-E_cond_xy + E_cond_x)
        grad_mu_trunc = (-E_cond_xy + E_cond_x)      # (batch,K)

        # analytic part: -grad_mu log p(y|x) (softmax term)
        # p(y|x) = softmax(mu)_y ; so grad_mu(-log p) = softmax(mu) - onehot(y)
        probs = ch.softmax(mu, dim=-1)
        idx = targ.squeeze(-1)
        onehot = ch.nn.functional.one_hot(idx, num_classes=mu.size(-1)).float().to(device)
        grad_mu_ce = (probs - onehot)                  # (batch,K)

        # total gradient w.r.t mu
        grad_mu = grad_mu_ce + grad_mu_trunc           # (batch,K)
        # multiply by upstream grad_output (batch,) -> (batch,1)
        grad_mu = grad_mu * grad_output.unsqueeze(1)

        # chain rule: if theta are parameters producing mu, caller must do mu.backward; here we return gradient for logits input.
        return grad_mu, None, None, None, None


In [50]:
trunc_ce_contained = TruncatedCE.apply

In [51]:
ch.manual_seed(69)
weight = ch.nn.Parameter(ch.randn(K, D))
optim = SGD([weight], lr=1e-2)


best_weight, best_loss = None, None

for i in range(epochs): 
    for batch in train_loader: 
        optim.zero_grad()
        x_, y_ = batch 
        
        logits = x_@weight.T 
        pred = softmax(logits)
        
        loss = trunc_ce_contained(logits, y_, phi)
        loss = loss.sum()
        
        if best_weight is None or loss < best_loss: 
            best_loss = loss 
            best_weight = weight.clone().detach()
        
        loss.backward()
        
        optim.step()
    print(f'Epoch {i+1}, loss={loss.item()}')
        
weight.requires_grad = False

filtered size: torch.Size([50, 1])
> /tmp/ipykernel_220710/2752663162.py(60)forward()
     58         # numerator and denominator for prob estimate
     59         import pdb; pdb.set_trace()
---> 60         num = (mask * filtered).sum(dim=0)                    # (batch, 1)
     61         den = filtered.sum(dim=0)                             # (batch, 1)
     62         prob_est = (num + eps) / (den + eps)                  # (batch, 1)

ipdb> p noised.size()
torch.Size([2000, 50, 2])
ipdb> phi(noised)
tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
  

In [37]:
print(f'estimated weight:\n {weight.data.tolist()}')
weight_diff_ = weight[1] - weight[0]
print(f'estimated weight diff:\n {weight_diff_.data.tolist()}')

estimated weight:
 [[0.5514603853225708, -1.9250638484954834, -0.7450966238975525], [0.35829824209213257, -1.2587908506393433, -0.625078022480011]]
estimated weight diff:
 [-0.19316214323043823, 0.6662729978561401, 0.1200186014175415]


In [38]:
soft_cos_sim = float(cos_sim(weight_diff_[None,...], W_eff))
pred = softmax(X@weight.T).argmax(-1)
soft_acc = pred.eq(y.flatten()).sum() / len(y)
print(f'softmax accuracy: {soft_acc}')
print(f'softmax cos sim: {soft_cos_sim}')
soft_conf_matrix = confusion_matrix(y, pred)
print(f'softmax confusion matrix: \n {soft_conf_matrix}')

softmax accuracy: 0.545799970626831
softmax cos sim: 0.9411816596984863
softmax confusion matrix: 
 [[ 176 2086]
 [ 185 2553]]
